In [3]:
import pandas as pd
import numpy as np
from utils.train_test_split import train_test_split
from utils.robust_scaler import RobustScaler

df = pd.read_csv("dataset/creditcard.csv")

# Drop duplicates
print(f"Before: {len(df)}")
df = df.drop_duplicates()
print(f"After: {len(df)}")

print("\n")

# Seperate Features and Label
y = df['Class']
X = df.drop('Class', axis=1)

print(f"X: {X.shape}")
print(f"y: {y.shape}")

print("\n")

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# Second split: 50% val, 50% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print(f"\n✓ Train: {X_train.shape} - Fraud: {y_train.mean()*100:.4f}% ({y_train.sum():,} cases)")
print(f"✓ Val:   {X_val.shape} - Fraud: {y_val.mean()*100:.4f}% ({y_val.sum():,} cases)")
print(f"✓ Test:  {X_test.shape} - Fraud: {y_test.mean()*100:.4f}% ({y_test.sum():,} cases)")

print("\n")
print(" BEFORE SCALING (raw Time & Amount dari training set)")
print(f" {'':<10s} {'Time':>12s} {'Amount':>12s}")
print(f" {'median':<10s} {X_train['Time'].median():>12.4f} {X_train['Amount'].median():>12.4f}")
print(f" {'IQR':<10s} {X_train['Time'].quantile(0.75) - X_train['Time'].quantile(0.25):>12.4f} {X_train['Amount'].quantile(0.75) - X_train['Amount'].quantile(0.25):>12.4f}")
print(f" {'min':<10s} {X_train['Time'].min():>12.4f} {X_train['Amount'].min():>12.4f}")
print(f" {'max':<10s} {X_train['Time'].max():>12.4f} {X_train['Amount'].max():>12.4f}")

# Create scaler
scaler = RobustScaler()

# FIT scaler on training data (learns median & IQR from train)
print(f"\n➤ Fitting RobustScaler on training data")
scaler.fit(X_train[['Time', 'Amount']])

# Transform all three sets using training statistics
X_train_scaled = X_train.copy()
X_val_scaled   = X_val.copy()
X_test_scaled  = X_test.copy()

X_train_scaled[['Time', 'Amount']] = scaler.transform(X_train[['Time', 'Amount']])
X_val_scaled[['Time', 'Amount']]   = scaler.transform(X_val[['Time', 'Amount']])
X_test_scaled[['Time', 'Amount']]  = scaler.transform(X_test[['Time', 'Amount']])

print("\n")
print(" AFTER SCALING (Time & Amount setelah RobustScaler)")
print(f" {'':<10s} {'Time':>12s} {'Amount':>12s}")
print(f" {'median':<10s} {X_train_scaled['Time'].median():>12.4f} {X_train_scaled['Amount'].median():>12.4f}")
print(f" {'IQR':<10s} {(X_train_scaled['Time'].quantile(0.75) - X_train_scaled['Time'].quantile(0.25)):>12.4f} {(X_train_scaled['Amount'].quantile(0.75) - X_train_scaled['Amount'].quantile(0.25)):>12.4f}")
print(f" {'min':<10s} {X_train_scaled['Time'].min():>12.4f} {X_train_scaled['Amount'].min():>12.4f}")
print(f" {'max':<10s} {X_train_scaled['Time'].max():>12.4f} {X_train_scaled['Amount'].max():>12.4f}")

# Add labels back
X_train_scaled['Class'] = y_train.values
X_val_scaled['Class']   = y_val.values
X_test_scaled['Class']  = y_test.values

# Save to separate CSV files
X_train_scaled.to_csv('dataset/creditcard_train.csv', index=False)
X_val_scaled.to_csv('dataset/creditcard_val.csv', index=False)
X_test_scaled.to_csv('dataset/creditcard_test.csv', index=False)

print("\n✓ Dataset berhasil disimpan ke folder dataset/")


Before: 284807
After: 283726


X: (283726, 30)
y: (283726,)



✓ Train: (198610, 30) - Fraud: 0.1672% (332 cases)
✓ Val:   (42559, 30) - Fraud: 0.1668% (71 cases)
✓ Test:  (42557, 30) - Fraud: 0.1645% (70 cases)


 BEFORE SCALING (raw Time & Amount dari training set)
                    Time       Amount
 median       84599.0000      22.1000
 IQR          85068.0000      72.2700
 min              0.0000       0.0000
 max         172792.0000   25691.1600

➤ Fitting RobustScaler on training data


 AFTER SCALING (Time & Amount setelah RobustScaler)
                    Time       Amount
 median           0.0000       0.0000
 IQR              1.0000       1.0000
 min             -0.9945      -0.3058
 max              1.0367     355.1828

✓ Dataset berhasil disimpan ke folder dataset/


In [4]:
import joblib

# Simpan scaler
joblib.dump(scaler, 'saved/robust_scaler.pkl')
print("✓ Scaler berhasil disimpan ke saved/robust_scaler.pkl")

✓ Scaler berhasil disimpan ke saved/robust_scaler.pkl
